# LocalAgent pretraining on a Colab GPU

**Pretraining** teaches a randomly initialized model to predict the next token across a broad corpus. It builds language, code, and world-pattern priors; it does not yet teach the final assistant protocol. LocalAgent therefore keeps canonical tool calls and trajectories for midtraining/SFT after this notebook's base-model stage.

This run uses a bounded, auditable mixture: 65% near-deduplicated FineWeb-Edu, 15% Cosmopedia v2, and 20% cleaned Python restricted to permissive per-file licenses. Raw data and shards stay on Colab's fast ephemeral disk. Only the tokenizer, manifests, exact runtime config, receipts, and resumable checkpoints are copied to Google Drive.

## 0. Select a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** (or L4/A100). Then run every cell in order. Free Colab hardware and session length are dynamic, so the notebook refuses to silently train on CPU.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from hashlib import sha256
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print({"torch": torch.__version__, "gpu": gpu_name, "gpu_GiB": round(gpu_gib, 1)})
subprocess.run(["nvidia-smi"], check=True)

## 1. Mount Drive and choose the run budget

`SMOKE=True` is the safe first execution: it proves GPU training and Drive persistence with a small corpus and 10 updates. Set it to `False` for the resumable 34M-parameter run. The full preset's 6,000 updates consume about 98M packed tokens—useful but still below the approximately 680M-token (20 tokens/parameter) first scaling target. Increase `TOTAL_STEPS` over later resumed sessions.

In [ ]:
from google.colab import drive
DRIVE_AVAILABLE = True
try:
    drive.mount("/content/drive")
except Exception as exc:
    DRIVE_AVAILABLE = False
    print("Drive mount was not authorized; artifacts will be downloaded at the end.")

SMOKE = True                 # change to False after the first stored checkpoint succeeds
TARGET_CHARS = 2_000_000 if SMOKE else 120_000_000
TOTAL_STEPS = 10 if SMOKE else 6_000
SEQ_LEN = 256 if SMOKE else 512
RUN_NAME = "webgpu-35m-smoke" if SMOKE else "webgpu-35m"

LOCAL_REPO = Path("/content/LocalAgent")
LOCAL_DATA = Path("/content/localagent-work")
LOCAL_RUN = LOCAL_DATA / "runs" / RUN_NAME
DRIVE_ROOT = Path("/content/drive/MyDrive/LocalAgent/pretraining") if DRIVE_AVAILABLE else Path("/content/localagent-export")
DRIVE_RUN = DRIVE_ROOT / RUN_NAME
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
print({"local_work": str(LOCAL_DATA), "persistent_target": str(DRIVE_RUN), "drive": DRIVE_AVAILABLE})

## 2. Install this repository

The default clones the public repository. If you are testing unpushed local edits, upload a project archive as `/content/LocalAgent.zip` first; this cell prefers that archive.

In [ ]:
archive = Path("/content/LocalAgent.zip")
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
if archive.exists():
    shutil.unpack_archive(str(archive), "/content")
    candidates = [p for p in Path("/content").glob("LocalAgent*") if p.is_dir()]
    extracted = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
    assert extracted is not None, "The uploaded archive does not contain pyproject.toml"
    if extracted != LOCAL_REPO:
        extracted.rename(LOCAL_REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/SangbumChoi/LocalAgent.git", str(LOCAL_REPO)
    ], check=True)

os.chdir(LOCAL_REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"], check=True)
git_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"], text=True, capture_output=True
).stdout.strip() if (LOCAL_REPO / ".git").exists() else "uploaded-working-tree"
print({"repo": str(LOCAL_REPO), "revision": git_revision})

## 3. Stream, filter, tokenize, and pack the corpus

The download is size-bounded and shuffled deterministically. Every accepted row keeps a stable ID, upstream dataset, repository/path when available, and license. Document-level hashing prevents train/validation leakage. The BPE tokenizer is trained only on this training corpus.

In [ ]:
raw_dir = LOCAL_DATA / "raw"
shards_dir = LOCAL_DATA / "shards"
tokenizer_path = LOCAL_DATA / "tokenizer-16k.json"
shutil.rmtree(raw_dir, ignore_errors=True)
shutil.rmtree(shards_dir, ignore_errors=True)
raw_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "scripts/download_pretrain_mixture.py",
    "configs/data/pretrain-colab.yaml", "--out", str(raw_dir),
    "--target-chars", str(TARGET_CHARS),
], check=True)
subprocess.run([
    sys.executable, "scripts/prepare_corpus.py", str(raw_dir / "mixture.jsonl"),
    "--out", str(shards_dir), "--seq-len", str(SEQ_LEN),
    "--rows-per-shard", "2048", "--val-fraction", "0.01",
    "--tokenizer", "bpe", "--vocab-size", "16384",
    "--tokenizer-path", str(tokenizer_path), "--seed", "42",
], check=True)

corpus_manifest = json.loads((shards_dir / "manifest.json").read_text())
download_manifest = json.loads((raw_dir / "download_manifest.json").read_text())
print({
    "documents": corpus_manifest["total_documents"],
    "packed_tokens": corpus_manifest["total_tokens"],
    "licenses": corpus_manifest["license_counts"],
})

## 4. Persist the immutable inputs and build the runtime config

Drive is used for a few large/important artifacts, not for live shard reads. If a checkpoint already exists in Drive, it is copied back to the VM and resumed. Raising `TOTAL_STEPS` continues from its stored `step`; lowering it does not rewind.

In [ ]:
import yaml

artifact_dir = DRIVE_RUN / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
for source, name in [
    (tokenizer_path, "tokenizer-16k.json"),
    (shards_dir / "manifest.json", "corpus_manifest.json"),
    (raw_dir / "download_manifest.json", "download_manifest.json"),
    (Path("configs/data/pretrain-colab.yaml"), "mixture_config.yaml"),
    (Path("configs/model/webgpu-35m-hybrid.yaml"), "model_config.yaml"),
]:
    shutil.copy2(source, artifact_dir / name)

LOCAL_RUN.mkdir(parents=True, exist_ok=True)
drive_checkpoint = DRIVE_RUN / "latest.pt"
local_checkpoint = LOCAL_RUN / "latest.pt"
if drive_checkpoint.exists():
    shutil.copy2(drive_checkpoint, local_checkpoint)
    print("Resuming checkpoint copied from Drive:", drive_checkpoint)

config = yaml.safe_load(Path("configs/train/pretrain-colab.yaml").read_text())
config["data"]["shards_dir"] = str(shards_dir)
config["data"]["tokenizer"]["path"] = str(tokenizer_path)
config["schedule"]["total_steps"] = TOTAL_STEPS
config["schedule"]["warmup_steps"] = min(100, max(1, TOTAL_STEPS // 20))
config["batch"]["micro_batch_size"] = 2 if gpu_gib >= 14 else 1
config["batch"]["grad_accum_steps"] = 2 if SMOKE else 16
config["log"]["out_dir"] = str(LOCAL_RUN)
config["log"]["mirror_dir"] = str(DRIVE_RUN)
config["log"]["ckpt_every"] = min(250, TOTAL_STEPS)
config["log"]["eval_every"] = min(250, TOTAL_STEPS)
runtime_config = LOCAL_DATA / "pretrain-runtime.yaml"
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
shutil.copy2(runtime_config, artifact_dir / "pretrain-runtime.yaml")
print(runtime_config.read_text())

## 5. Train on CUDA and mirror checkpoints to Drive

Loss should begin near `ln(16384) ≈ 9.70` for random logits and trend downward. A tiny smoke run is only an infrastructure proof; it does not produce a useful language model.

In [ ]:
started_at = time.time()
subprocess.run(["localagent", "model-info", "configs/model/webgpu-35m-hybrid.yaml"], check=True)
subprocess.run(["localagent", "train", "pretrain", str(runtime_config)], check=True)
elapsed = time.time() - started_at
torch.cuda.synchronize()
print({"elapsed_seconds": round(elapsed, 1), "peak_cuda_GiB": round(torch.cuda.max_memory_allocated() / 2**30, 2)})

## 6. Verify the stored checkpoint and write a receipt

This final cell fails if Drive does not contain a loadable checkpoint. The receipt makes the result independently inspectable after the Colab VM disappears.

In [ ]:
assert drive_checkpoint.exists(), f"Missing Drive checkpoint: {drive_checkpoint}"
saved = torch.load(drive_checkpoint, map_location="cpu", weights_only=False)
checkpoint_sha = sha256(drive_checkpoint.read_bytes()).hexdigest()
receipt = {
    "completed_at_unix": time.time(),
    "git_revision": git_revision,
    "gpu": gpu_name,
    "gpu_memory_GiB": round(gpu_gib, 2),
    "torch": torch.__version__,
    "smoke": SMOKE,
    "elapsed_seconds": elapsed,
    "checkpoint": str(drive_checkpoint),
    "checkpoint_sha256": checkpoint_sha,
    "step": int(saved["step"]),
    "tokens_seen": int(saved["tokens_seen"]),
    "final_loss": float(saved["loss_history"][-1]),
    "packed_corpus_tokens": int(corpus_manifest["total_tokens"]),
    "corpus_manifest_sha256": sha256((shards_dir / "manifest.json").read_bytes()).hexdigest(),
    "runtime_config": config,
}
(DRIVE_RUN / "run_receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
print("VERIFIED AND STORED:", DRIVE_RUN)
if not DRIVE_AVAILABLE:
    import shutil
    from google.colab import files
    bundle = shutil.make_archive(f"/content/{RUN_NAME}-artifacts", "zip", DRIVE_RUN)
    print("Downloading fallback bundle:", bundle)
    files.download(bundle)